In [1]:
import time
import pandas as pd
import random
import requests
from lxml import etree
import csv

In [2]:
headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36 Edg/135.0.0.0',
}

In [3]:
def style_name_get():
    url = 'https://www.dongchedi.com'
    style_name_response = requests.get(url, headers=headers)
    style_name_response.close()

    style_name_page_tree = etree.HTML(style_name_response.text)
    div_list = style_name_page_tree.xpath('//*[@id="__next"]/div/div[2]/div[1]/div/div/div[1]/div[1]')

    for div in div_list:
        style_name_list = div.xpath('.//a//text()')
        href_list = [url + href for href in div.xpath('.//a/@href')]

    return style_name_list, href_list

In [4]:
style_name_list, href_list = style_name_get()
print(style_name_list, href_list)

['新能源', '轿车', 'SUV', 'MPV', '跑车', '皮卡', '微面', '微卡', '轻客'] ['https://www.dongchedi.com/auto/library/x-x-x-x-x-x-4-x-x-x-x', 'https://www.dongchedi.com/auto/library/x-0-x-x-x-x-x-x-x-x-x', 'https://www.dongchedi.com/auto/library/x-1-x-x-x-x-x-x-x-x-x', 'https://www.dongchedi.com/auto/library/x-2-x-x-x-x-x-x-x-x-x', 'https://www.dongchedi.com/auto/library/x-4-x-x-x-x-x-x-x-x-x', 'https://www.dongchedi.com/auto/library/x-3-x-x-x-x-x-x-x-x-x', 'https://www.dongchedi.com/auto/library/x-7-x-x-x-x-x-x-x-x-x', 'https://www.dongchedi.com/auto/library/x-8-x-x-x-x-x-x-x-x-x', 'https://www.dongchedi.com/auto/library/x-6-x-x-x-x-x-x-x-x-x']


In [7]:
def car_id_get():
    new_url = 'https://www.dongchedi.com/motor/pc/car/brand/select_series_v2?aid=1839&app_name=auto_web_pc'
    car_ids = []

    for i in range(1):
        temp = []
        flag = 0
        series_type = 6
        for j in range(1, 200):
            data = {
                'series_type': series_type,
                'sort_new': 'hot_desc',
                'city_name': '重庆',
                'limit': 30,
                'page': j,
            }
            new_res = requests.post(new_url, headers=headers, data=data)
            new_res.close()

            for series in new_res.json()['data']['series']:
                temp.append(series['id'])

            if flag == temp[-1]:
                car_ids.append(temp)
                print('ok!')
                print(len(temp))
                break
            else:
                flag = temp[-1]

            print(temp[-1], j)

            time.sleep(random.randint(2, 3))

    return car_ids

In [8]:
car_ids = car_id_get()
print(car_ids)
print(len(car_ids))

9464 1
24548 2
9429 3
4212 4
9577 5
9557 6
ok!
153
[[575, 6052, 1213, 6252, 9241, 1418, 574, 1777, 9503, 247, 10163, 25148, 2972, 8804, 25093, 782, 549, 3441, 25064, 6278, 594, 803, 777, 25328, 6268, 4715, 9434, 6094, 1486, 9464, 3742, 24879, 426, 802, 1597, 25233, 25351, 595, 804, 1825, 9777, 9458, 1487, 25200, 1420, 9782, 24740, 2927, 4708, 25352, 9247, 6180, 25097, 5964, 2921, 3767, 826, 4623, 5247, 24548, 9810, 834, 1826, 9809, 4179, 427, 3718, 25114, 24739, 25626, 8849, 825, 5452, 1830, 4767, 836, 829, 6028, 6029, 9974, 3180, 9921, 831, 1838, 25649, 4660, 4562, 3499, 9674, 9429, 1831, 3012, 833, 9284, 9569, 9289, 2521, 2023, 4008, 25463, 3019, 5261, 659, 1924, 3957, 2174, 2796, 4641, 9201, 4198, 4793, 4809, 24912, 3029, 1828, 4310, 9654, 10007, 9308, 4212, 5591, 5260, 4187, 5310, 25411, 4059, 4559, 3031, 2628, 9753, 4621, 2642, 25304, 4936, 4165, 9578, 5182, 9580, 5116, 9514, 9576, 9516, 3082, 4356, 1718, 9526, 9515, 4186, 9525, 9577, 4557, 9556, 9557]]
1


In [9]:
def car_info(car_id):
    url = 'https://www.dongchedi.com/auto/params-carIds-x-' + str(car_id)
    res = requests.get(url, headers=headers)
    res.close()

    info_tree = etree.HTML(res.text)

    # 车名获取
    car_name_list = []
    div_list = info_tree.xpath('//*[@id="__next"]/div/div/div/div[2]/div[2]/div[1]/div[1]/div')
    for div in div_list[1:]:
        car_name = div.xpath('.//a//text()')
        car_name_list += car_name
    car_info_list = [{'车名': car_name} for car_name in car_name_list]


    # 其他信息获取
    num = len(info_tree.xpath('//*[@id="__next"]/div/div/div/div[2]/div[1]/div/ul/li'))
    for i in range(num):
        div_list = info_tree.xpath(f'//*[@id="__next"]/div/div/div/div[2]/div[2]/div[{i + 2}]/div')
        num_1 = len(div_list)

        for j in range(num_1):
            if j == 0:
                title = div_list[j].xpath('.//h3/text()')[0]
                for q in car_info_list:
                    q[title] = {}

            else:
                new_title = div_list[j].xpath('.//label/text()')
                info_list = []
                new_div_list = div_list[j]
                for new_div in new_div_list[1:]:
                    if new_div.xpath('.//text()') == []:
                        info_list.append('-')
                    elif new_div.xpath('.//text()')[0] == '●':
                        info_list.append(new_div.xpath('.//span/following-sibling::text()')[0])
                    elif new_div.xpath('.//text()')[0] == '○':
                        info_list.append(new_div.xpath('.//span/following-sibling::text()')[0])
                    else:
                        info_list.append(new_div.xpath('.//text()')[0])
                if len(info_list) == 1:
                    info_list *= len(car_name_list)
                # print(new_title + info_list)
                for k in range(len(car_info_list)):
                    car_info_list[k][title][new_title[0]] = info_list[k]

    return car_info_list

In [11]:
all_info = []
j = 1
for car_id_list in car_ids:
    for car_id in car_id_list:
        all_info += car_info(car_id)
        print(car_id, f' 进度：{j}/{len(car_id_list)}')
        j += 1
        time.sleep(random.randint(1, 3))
    j = 1
    break
print(len(all_info))
print(all_info)

575  进度：1/153
6052  进度：2/153
1213  进度：3/153
6252  进度：4/153
9241  进度：5/153
1418  进度：6/153
574  进度：7/153
1777  进度：8/153
9503  进度：9/153
247  进度：10/153
10163  进度：11/153
25148  进度：12/153
2972  进度：13/153
8804  进度：14/153
25093  进度：15/153
782  进度：16/153
549  进度：17/153
3441  进度：18/153
25064  进度：19/153
6278  进度：20/153
594  进度：21/153
803  进度：22/153
777  进度：23/153
25328  进度：24/153
6268  进度：25/153
4715  进度：26/153
9434  进度：27/153
6094  进度：28/153
1486  进度：29/153
9464  进度：30/153
3742  进度：31/153
24879  进度：32/153
426  进度：33/153
802  进度：34/153
1597  进度：35/153
25233  进度：36/153
25351  进度：37/153
595  进度：38/153
804  进度：39/153
1825  进度：40/153
9777  进度：41/153
9458  进度：42/153
1487  进度：43/153
25200  进度：44/153
1420  进度：45/153
9782  进度：46/153
24740  进度：47/153
2927  进度：48/153
4708  进度：49/153
25352  进度：50/153
9247  进度：51/153
6180  进度：52/153
25097  进度：53/153
5964  进度：54/153
2921  进度：55/153
3767  进度：56/153
826  进度：57/153
4623  进度：58/153
5247  进度：59/153
24548  进度：60/153
9810  进度：61/153
834  进度：62/153
1826  进度：63/153
98

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [12]:
print(len(all_info))
print(all_info[0])

1411
{'车名': '全顺 2024款 T8 2.0T柴油手动短轴中顶3座货运版', '基本信息': {'官方指导价': '15.38万', '厂商': '江铃福特', '级别': '轻客', '能源类型': '柴油', '上市时间': '2023.10', '发动机': '2.0T 154马力 L4', '最大功率(kW)': '113(154Ps)', '最大扭矩(N·m)': '390', '变速箱': '6挡手动', '长x宽x高(mm)': '5230x2068x2485', '车身结构': '5门3座货车', '最高车速(km/h)': '-', '整车保修期限': '3年或6万公里'}, '车身': {'长(mm)': '5230', '宽(mm)': '2068', '高(mm)': '2485', '轴距(mm)': '3000', '前轮距(mm)': '-', '后轮距(mm)': '-', '最小离地间隙(mm)': '-', '车身结构': '货车', '车门数(个)': '5', '车门开启方式': '侧滑门', '座位数(个)': '3', '整备质量(kg)': '-', '满载质量(kg)': '-', '油箱容积(L)': '80.0', '行李舱容积(L)': '-', '最小转弯半径': '-'}, '发动机': {'发动机型号': 'DURATORQ4D20D6H', '排量(mL)': '2000', '排量(L)': '2.0', '进气形式': '涡轮增压', '气缸排列形式': 'L', '气缸数(个)': '4', '每缸气门数(个)': '4', '压缩比': '-', '配气机构': 'DOHC', '最大马力(Ps)': '154', '最大功率(kW)': '113', '最大净功率(kW)': '-', '最大功率转速(rpm)': '3400', '最大扭矩(N·m)': '390', '最大扭矩转速(rpm)': '1500-2500', '发动机特有技术': '-', '燃料形式': '柴油', '燃油标号': '0#', '供油方式': '柴油直喷', '缸盖材料': '铝合金', '缸体材料': '未知', '环保标准': '国VI b'}, '变速箱': {'变速箱描述': '6挡手动',

In [13]:
import json
with open(r'D:\pycharm_files\人工智能\car_select_helper\car_info\轻客.json', 'w', encoding='utf-8') as f:
    json.dump(all_info, f, ensure_ascii=False, indent=4)